# UAV Battery Tool — Notebook 05: Report Generation

Generate formatted Excel reports from simulation results, battery comparisons, and flight log analysis.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import warnings; warnings.filterwarnings('ignore')

from batteries.database import BatteryDatabase
from batteries.log_importer import generate_synthetic_log
from batteries.parameter_fitter import fit_all, apply_fitted_params
from mission.simulator import run_simulation, compare_batteries, temperature_sweep
from mission.report_generator import generate_report

DB_PATH = '../battery_db.xlsx'
db = BatteryDatabase(DB_PATH).load()
print(db.summary())

═══ Battery Database Summary ═══
  Chemistries       : 9
  Cells             : 11
  Battery packs     : 8
  Discharge points  : 132
  Equipment items   : 29
  UAV configurations: 3
  Mission profiles  : 3


## 1 · Configure Report Parameters

In [2]:
# ── Load settings from 00_configurator.ipynb (analysis_config.json) ───────────
import json as _json, os as _os
_CFG_PATH = _os.path.join(_os.path.dirname(_os.path.abspath('.')), 'analysis_config.json')
if not _os.path.exists(_CFG_PATH):
    _CFG_PATH = 'analysis_config.json'
_cfg = {}
if _os.path.exists(_CFG_PATH):
    with open(_CFG_PATH) as _f:
        _cfg = _json.load(_f)
    print(f'Loaded config from {_CFG_PATH}')
else:
    print('No analysis_config.json found — using defaults below (run 00_configurator first)')

# ── Values from configurator (overridden by manual entries below if desired) ───
_sel_bats = _cfg.get('selected_batteries', ['BAT_MID_6S2P', 'BAT_MID_6S4P', 'BAT_AG_6S1P'])
PRIMARY_PACK_ID  = _sel_bats[0] if isinstance(_sel_bats, list) and _sel_bats else 'BAT_MID_6S2P'
COMPARE_PACK_IDS = _sel_bats if isinstance(_sel_bats, list) else ['BAT_MID_6S2P', 'BAT_MID_6S4P', 'BAT_AG_6S1P']
MISSION_ID       = _cfg.get('mission_id',     'SURVEY_STD')
UAV_ID           = _cfg.get('uav_id',         'HEX_SURVEY_900')
AMBIENT_TEMP_C   = _cfg.get('ambient_temp_c', 25.0)
TEMP_SWEEP       = _cfg.get('temp_sweep',     [-25, -10, 0, 15, 25, 40])


# ── Reconstruct combined pack if configured ────────────────────────────────────
_combo_cfg = _cfg.get('battery_combination')
if _combo_cfg:
    from batteries.builder import combine_packs as _combine_packs
    _combo_packs = [db.packs[bid] for bid in _combo_cfg.get('packs', []) if bid in db.packs]
    if len(_combo_packs) >= 2:
        _combined = _combine_packs(_combo_packs, topology=_combo_cfg.get('topology', 'series'))
        db.packs[_combined.battery_id] = _combined
        print(f'Combined pack registered: {_combined.battery_id}  '
              f'({_combined.pack_voltage_nom:.1f}V {_combined.pack_energy_wh:.0f}Wh)')

# ── Manual overrides (uncomment to override configurator values) ────────────
# ── Select packs, mission, and UAV ────────────────────────────────────────────
# PRIMARY_PACK_ID    = 'BAT_MID_6S2P'
# COMPARE_PACK_IDS   = ['BAT_MID_6S2P', 'BAT_MID_6S4P', 'BAT_AG_6S1P']
# MISSION_ID         = 'SURVEY_STD'
# UAV_ID             = 'HEX_SURVEY_900'
# AMBIENT_TEMP_C     = 25.0
# TEMP_SWEEP         = [-20,-15,-10,-5,0,5,10,15,20,25,30,35,40,45]
INCLUDE_LOG        = True   # generate synthetic log and fit parameters
OUT_PATH           = 'UAV_Battery_Report.xlsx'
# ─────────────────────────────────────────────────────────────────────────────

primary_pack = db.packs[PRIMARY_PACK_ID]
mission      = db.missions[MISSION_ID]
uav          = db.uav_configs[UAV_ID]
compare_packs= [db.packs[pid] for pid in COMPARE_PACK_IDS if pid in db.packs]
print(f'Primary pack  : {primary_pack}')
print(f'Compare packs : {[p.battery_id for p in compare_packs]}')
print(f'Mission       : {mission}')

Primary pack  : BAT_MID_6S2P: Mid UAV 6S2P 21700 | 6S2P | 21.9V 9.0Ah (197Wh) | 840g
Compare packs : ['BAT_MID_6S2P', 'BAT_MID_6S4P', 'BAT_AG_6S1P']
Mission       : Mission 'Grid Survey 500m' (SURVEY_STD): 8 phases | 24.5 min | 6700m


## 2 · Run Simulations

In [3]:
print('Running simulations...')

# Primary simulation
primary_result = run_simulation(
    pack=primary_pack, mission=mission, uav=uav,
    discharge_pts=db.discharge_pts,
    ambient_temp_c=AMBIENT_TEMP_C, dt_s=1.0
)
print(primary_result.summary())

# Multi-pack comparison
compare_results = compare_batteries(
    packs=compare_packs, mission=mission, uav=uav,
    discharge_pts=db.discharge_pts,
    ambient_temp_c=AMBIENT_TEMP_C, dt_s=2.0
)
print(f'\nMulti-pack results:')
for r in compare_results:
    print(f'  {r.pack_id}: SoC={r.final_soc:.1f}%  Vmin={r.min_voltage:.3f}V  depleted={r.depleted}')

Running simulations...
════════════════════════════════════════════════════
 Simulation: BAT_MID_6S2P × SURVEY_STD  [COMPLETED]
════════════════════════════════════════════════════
  Duration         : 1469 s  (24.5 min)
  Energy consumed  : 127.91 Wh
  Initial SoC      : 100.0 %
  Final SoC        : 34.0 %
  Min voltage      : 19.657 V
  Max current      : 30.7 A
  Peak sag total   : 1.537 V
  Peak temperature : 30.7 °C

Multi-pack results:
  BAT_MID_6S2P: SoC=34.0%  Vmin=19.661V  depleted=False
  BAT_MID_6S4P: SoC=71.2%  Vmin=19.257V  depleted=False
  BAT_AG_6S1P: SoC=33.5%  Vmin=19.113V  depleted=False


In [4]:
# Temperature sweep
print(f'Running temperature sweep ({len(TEMP_SWEEP)} temperatures)...')
sweep_results = temperature_sweep(
    pack=primary_pack, mission=mission, uav=uav,
    discharge_pts=db.discharge_pts,
    temperatures_c=TEMP_SWEEP, dt_s=5.0
)
print('Done.')

df_sweep = pd.DataFrame([{
    'Temp (C)': t, 'Final SoC (%)': round(r.final_soc,1),
    'Peak sag (V)': round(r.peak_sag_v,3), 'Min V (V)': round(r.min_voltage,3),
    'Max T (C)': round(r.max_temp_c,1), 'Depleted': r.depleted}
    for t, r in zip(TEMP_SWEEP, sweep_results)])
print(df_sweep.to_string(index=False))

Running temperature sweep (14 temperatures)...
Done.
 Temp (C)  Final SoC (%)  Peak sag (V)  Min V (V)  Max T (C)  Depleted
      -20           10.0        14.529     11.428        0.4      True
      -15           11.4         7.722     16.401        2.3     False
      -10           16.2         5.672     17.325        4.9     False
       -5           20.4         4.422     17.903        7.9     False
        0           24.3         3.559     18.367       11.1     False
        5           27.2         2.926     18.740       14.7     False
       10           29.7         2.446     19.056       18.4     False
       15           31.9         2.072     19.332       22.4     False
       20           33.5         1.776     19.555       26.5     False
       25           34.0         1.537     19.674       30.7     False
       30           33.4         1.342     19.702       35.1     False
       35           32.5         1.181     19.699       39.6     False
       40           31.3

## 3 · Flight Log Analysis (optional)

In [5]:
log = fitted = None
if INCLUDE_LOG:
    from batteries.log_importer import generate_synthetic_log
    from batteries.parameter_fitter import fit_all, apply_fitted_params
    print('Generating synthetic flight log and fitting parameters...')
    log = generate_synthetic_log(primary_pack, mission, uav, db.discharge_pts,
                                  ambient_temp_c=AMBIENT_TEMP_C, dt_s=2.0,
                                  noise_v=0.03, noise_i=0.8)
    fitted = fit_all(log, primary_pack.pack_capacity_ah,
                     primary_pack.pack_voltage_cutoff,
                     chem_id=primary_pack.chemistry_id,
                     pack_id=primary_pack.battery_id)
    print(fitted.summary())
else:
    print('Skipping log analysis (INCLUDE_LOG=False)')

Generating synthetic flight log and fitting parameters...
  [1/5] Fitting internal resistance...
        R_internal_mohm: 48.3790 ± 14.9090  R²=0.922  n=716
  [2/5] Reconstructing OCV curve...
        11 OCV points  R²=0.064
  [3/5] Fitting Peukert exponent...
        peukert_k: 1.0000 ± 0.0200  R²=0.000  n=733
  [4/5] Fitting Arrhenius temperature coefficients...
        B_ohmic_K: 0.0000 ± 0.0000  R²=0.000  n=0
        B_ct_K: 0.0000 ± 0.0000  R²=0.000  n=0
  [5/5] Estimating actual capacity...
        actual_capacity_ah: 9.0000 ± 0.0000  R²=0.000  n=0
══ Fitted Battery Parameters ══
  R_internal   : R_internal_mohm: 48.3790 ± 14.9090  R²=0.922  n=716
  Capacity     : actual_capacity_ah: 9.0000 ± 0.0000  R²=0.000  n=0
  Peukert k    : peukert_k: 1.0000 ± 0.0200  R²=0.000  n=733
  B_ohmic      : B_ohmic_K: 0.0000 ± 0.0000  R²=0.000  n=0
  B_ct         : B_ct_K: 0.0000 ± 0.0000  R²=0.000  n=0
  OCV curve    : 11 points fitted


## 4 · Generate Excel Report

In [6]:
print(f'Generating report: {OUT_PATH}')
report_path = generate_report(
    out_path=OUT_PATH,
    results=compare_results,
    packs=compare_packs,
    mission=mission,
    uav_name=UAV_ID,
    ambient_temp_c=AMBIENT_TEMP_C,
    temp_sweep_temps=TEMP_SWEEP,
    temp_sweep_results=sweep_results,
    flight_log=log,
    fitted_params=fitted,
    primary_pack=primary_pack,
)
print(f'Report saved: {report_path}')

Generating report: UAV_Battery_Report.xlsx
✓ Report saved → UAV_Battery_Report.xlsx
Report saved: UAV_Battery_Report.xlsx


## 4b · Generate PDF Summary Report

A beautiful multi-page PDF with cover, mission timeline, battery scorecard, temperature sweep, and simulation charts.

In [7]:
import datetime
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np

# ── Configuration ──────────────────────────────────────────────────────────────
PDF_OUT_PATH  = 'UAV_Battery_Report.pdf'   # output filename
COMPANY_NAME  = 'Pen Aviation Sdn Bhd'     # shown on cover

PHASE_COLORS = {
    'IDLE':            '#AAAAAA',  # ground idle
    'TAKEOFF':         '#FF9944',  # full power lift-off
    'CLIMB':           '#FFCC44',  # powered climb
    'CRUISE':          '#44AA66',  # level cruise
    'HOVER':           '#4488FF',  # stationary hover
    'DESCEND':         '#88AADD',  # descent
    'LAND':            '#CC88DD',  # landing
    'PAYLOAD_OPS':     '#FF6688',  # payload operations
    'EMERGENCY':       '#FF2222',  # emergency
    # ── Fixed-wing VTOL phases ────────────────────────────────────────
    'VTOL_TRANSITION': '#FF6611',  # lift+thrust overlap during transition
    'VTOL_HOVER':      '#22AAFF',  # explicit multirotor hover
    'FW_CRUISE':       '#00CC77',  # efficient fixed-wing cruise
    'FW_CLIMB':        '#AACC44',  # fixed-wing climb
    'FW_DESCEND':      '#99CCEE',  # glide descent
}
CHEM_COLORS  = {'LIPO':'#FFB347','LION21':'#4488FF','LIFEPO4':'#66BB6A',
                 'SSS':'#AB47BC','LITO':'#26C6DA','LION':'#FF7043','LIHV':'#FF6E40'}

DARK  = '#1A237E'   # deep navy
ACCENT= '#2196F3'   # blue
LIGHT = '#F5F7FA'   # near-white background
TEXT  = '#212121'

plt.rcParams.update({'font.family': 'DejaVu Sans', 'axes.grid': True,
                     'grid.alpha': 0.2, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.facecolor': 'white'})

def _header(fig, title, subtitle='', page_num=None):
    """Draw a consistent navy header band across the top of a figure."""
    ax_h = fig.add_axes([0, 0.93, 1, 0.07])
    ax_h.set_facecolor(DARK); ax_h.set_xlim(0,1); ax_h.set_ylim(0,1)
    ax_h.axis('off')
    ax_h.text(0.02, 0.55, title, color='white', fontsize=14, fontweight='bold', va='center')
    if subtitle:
        ax_h.text(0.02, 0.15, subtitle, color='#90CAF9', fontsize=9, va='center')
    if page_num:
        ax_h.text(0.98, 0.5, str(page_num), color='#90CAF9', fontsize=9,
                  ha='right', va='center')
    ax_h.axhline(0, color=ACCENT, linewidth=3)

def _footer(fig, left='', right=''):
    ax_f = fig.add_axes([0, 0, 1, 0.035])
    ax_f.set_facecolor('#ECEFF1'); ax_f.set_xlim(0,1); ax_f.set_ylim(0,1)
    ax_f.axis('off')
    ax_f.text(0.02, 0.5, left,  color='#607D8B', fontsize=7, va='center')
    ax_f.text(0.98, 0.5, right, color='#607D8B', fontsize=7, va='center', ha='right')

def _pill(ax, x, y, text, color, width=0.18, height=0.08):
    from matplotlib.patches import FancyBboxPatch
    box = FancyBboxPatch((x-width/2, y-height/2), width, height,
                          boxstyle='round,pad=0.01', facecolor=color,
                          edgecolor='none', transform=ax.transAxes, clip_on=False,
                          zorder=5)
    ax.add_patch(box)
    ax.text(x, y, text, transform=ax.transAxes, ha='center', va='center',
            color='white', fontsize=8, fontweight='bold', zorder=6)

now_str = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
run_info = (f'Mission: {MISSION_ID}  |  UAV: {UAV_ID}  |  '
            f'Temp: {AMBIENT_TEMP_C}°C  |  Generated: {now_str}')

with PdfPages(PDF_OUT_PATH) as pdf:

    # ══════════════════════════════════════════════════════════════════════════
    # PAGE 1 — COVER
    # ══════════════════════════════════════════════════════════════════════════
    fig = plt.figure(figsize=(11.69, 8.27))  # A4 landscape
    fig.patch.set_facecolor(DARK)

    # Large title block
    fig.text(0.5, 0.72, 'UAV Battery Analysis', color='white',
             fontsize=38, fontweight='bold', ha='center', va='center')
    fig.text(0.5, 0.62, 'Performance, Simulation & Scorecard Report',
             color='#90CAF9', fontsize=18, ha='center', va='center')

    # Accent bar
    ax_bar = fig.add_axes([0.15, 0.56, 0.70, 0.004])
    ax_bar.set_facecolor(ACCENT); ax_bar.axis('off')

    # Meta block
    meta = (f'{COMPANY_NAME}\n\n'
            f'Mission          {MISSION_ID}\n'
            f'UAV config       {UAV_ID}\n'
            f'Primary pack     {PRIMARY_PACK_ID}\n'
            f'Temperature      {AMBIENT_TEMP_C}°C\n'
            f'Generated        {now_str}')
    fig.text(0.5, 0.35, meta, color='#CFD8DC', fontsize=12,
             ha='center', va='center', linespacing=1.8,
             fontfamily='monospace')

    # Battery count pill
    ax_pill = fig.add_axes([0.35, 0.08, 0.30, 0.10])
    ax_pill.set_facecolor(ACCENT); ax_pill.axis('off')
    ax_pill.text(0.5, 0.6, f'{len(compare_packs)} batteries analysed',
                 color='white', fontsize=14, fontweight='bold',
                 ha='center', va='center', transform=ax_pill.transAxes)
    ax_pill.text(0.5, 0.2, f'across {len(TEMP_SWEEP)} temperature points',
                 color='#E3F2FD', fontsize=10,
                 ha='center', va='center', transform=ax_pill.transAxes)

    pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    # ══════════════════════════════════════════════════════════════════════════
    # PAGE 2 — MISSION PROFILE + EQUIPMENT SUMMARY
    # ══════════════════════════════════════════════════════════════════════════
    fig = plt.figure(figsize=(11.69, 8.27))
    _header(fig, 'Mission Profile', run_info, 'p.2')
    _footer(fig, COMPANY_NAME, PDF_OUT_PATH)

    gs = gridspec.GridSpec(2, 2, figure=fig, left=0.06, right=0.97,
                           top=0.89, bottom=0.08, hspace=0.45, wspace=0.35)

    # Mission timeline (Gantt)
    ax_gantt = fig.add_subplot(gs[0, :])
    x_pos = 0
    for ph in mission.phases:
PHASE_COLORS = {
    'IDLE':            '#AAAAAA',  # ground idle
    'TAKEOFF':         '#FF9944',  # full power lift-off
    'CLIMB':           '#FFCC44',  # powered climb
    'CRUISE':          '#44AA66',  # level cruise
    'HOVER':           '#4488FF',  # stationary hover
    'DESCEND':         '#88AADD',  # descent
    'LAND':            '#CC88DD',  # landing
    'PAYLOAD_OPS':     '#FF6688',  # payload operations
    'EMERGENCY':       '#FF2222',  # emergency
    # ── Fixed-wing VTOL phases ────────────────────────────────────────
    'VTOL_TRANSITION': '#FF6611',  # lift+thrust overlap during transition
    'VTOL_HOVER':      '#22AAFF',  # explicit multirotor hover
    'FW_CRUISE':       '#00CC77',  # efficient fixed-wing cruise
    'FW_CLIMB':        '#AACC44',  # fixed-wing climb
    'FW_DESCEND':      '#99CCEE',  # glide descent
}
        ax_gantt.barh(0, ph.duration_s, left=x_pos, height=0.5,
                      color=color, edgecolor='white', linewidth=0.5)
        if ph.duration_s > 30:
            ax_gantt.text(x_pos + ph.duration_s/2, 0,
                          f'{ph.phase_type}\n{ph.duration_s}s',
                          ha='center', va='center', fontsize=8,
                          fontweight='bold', color='white')
        x_pos += ph.duration_s
    ax_gantt.set_xlim(0, x_pos); ax_gantt.set_yticks([])
    ax_gantt.set_xlabel('Time (s)'); ax_gantt.set_title('Mission Phase Timeline')

    # Phase duration pie
    ax_pie = fig.add_subplot(gs[1, 0])
    durations = [ph.duration_s for ph in mission.phases]
    labels    = [ph.phase_type for ph in mission.phases]
PHASE_COLORS = {
    'IDLE':            '#AAAAAA',  # ground idle
    'TAKEOFF':         '#FF9944',  # full power lift-off
    'CLIMB':           '#FFCC44',  # powered climb
    'CRUISE':          '#44AA66',  # level cruise
    'HOVER':           '#4488FF',  # stationary hover
    'DESCEND':         '#88AADD',  # descent
    'LAND':            '#CC88DD',  # landing
    'PAYLOAD_OPS':     '#FF6688',  # payload operations
    'EMERGENCY':       '#FF2222',  # emergency
    # ── Fixed-wing VTOL phases ────────────────────────────────────────
    'VTOL_TRANSITION': '#FF6611',  # lift+thrust overlap during transition
    'VTOL_HOVER':      '#22AAFF',  # explicit multirotor hover
    'FW_CRUISE':       '#00CC77',  # efficient fixed-wing cruise
    'FW_CLIMB':        '#AACC44',  # fixed-wing climb
    'FW_DESCEND':      '#99CCEE',  # glide descent
}
    ax_pie.pie(durations, labels=labels, colors=colors_p, autopct='%1.0f%%',
               startangle=90, textprops={'fontsize': 8})
    ax_pie.set_title('Phase breakdown')

    # Simulated SoC + voltage trace
    ax_sv = fig.add_subplot(gs[1, 1])
    t_arr = np.array(primary_result.time_s)
    ax_sv.plot(t_arr, primary_result.soc_pct, ACCENT, linewidth=2, label='SoC (%)')
    ax2 = ax_sv.twinx()
    ax2.plot(t_arr, primary_result.voltage_v, '#E53935', linewidth=1.5,
             linestyle='--', alpha=0.8, label='Voltage (V)')
    ax_sv.set_xlabel('Time (s)'); ax_sv.set_ylabel('SoC (%)', color=ACCENT)
    ax2.set_ylabel('Voltage (V)', color='#E53935')
    ax_sv.set_title(f'{PRIMARY_PACK_ID} — SoC & Voltage')
    ax_sv.set_ylim(0, 110)

    pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    # ══════════════════════════════════════════════════════════════════════════
    # PAGE 3 — BATTERY SCORECARD
    # ══════════════════════════════════════════════════════════════════════════
    fig = plt.figure(figsize=(11.69, 8.27))
    _header(fig, 'Battery Scorecard', run_info, 'p.3')
    _footer(fig, COMPANY_NAME, PDF_OUT_PATH)

    ax_sc = fig.add_axes([0.04, 0.08, 0.92, 0.80])
    ax_sc.axis('off')

    col_labels = ['Battery', 'Chemistry', 'Energy\n(Wh)', 'Weight\n(g)',
                  'Wh/kg', 'Final SoC\n(%)', 'Min V\n(V)',
                  'Peak sag\n(V)', 'Max I\n(A)', 'Status']
    col_widths = [0.18, 0.09, 0.07, 0.07, 0.06, 0.09, 0.07, 0.08, 0.07, 0.09]

    # Header row
    y = 0.95; row_h = 0.075
    x = 0.0
    for lbl, w in zip(col_labels, col_widths):
        ax_sc.text(x + w/2, y, lbl, ha='center', va='center', fontsize=8,
                   fontweight='bold', color='white',
                   bbox=dict(boxstyle='square,pad=0.3', facecolor=DARK, edgecolor='none'))
        x += w

    # Data rows
    for idx, (r, p) in enumerate(zip(compare_results, compare_packs)):
        y -= row_h
        bg = '#FAFAFA' if idx % 2 == 0 else 'white'
        status = 'PASS' if not r.depleted and r.final_soc > 15 else \
                 ('MARGINAL' if not r.depleted else 'FAIL')
        status_color = {'PASS': '#2E7D32', 'MARGINAL': '#F57F17', 'FAIL': '#C62828'}[status]
        vals = [p.battery_id, p.chemistry_id, f'{p.pack_energy_wh:.0f}',
                f'{p.pack_weight_g:.0f}',
                f'{getattr(p, "specific_energy_wh_kg", p.pack_energy_wh/p.pack_weight_g*1000):.0f}',
                f'{r.final_soc:.1f}', f'{r.min_voltage:.3f}',
                f'{r.peak_sag_v:.3f}', f'{r.max_current:.1f}', status]
        x = 0.0
        for v, w in zip(vals, col_widths):
            color = status_color if v == status else TEXT
            fw    = 'bold' if v == status else 'normal'
            ax_sc.text(x + w/2, y + row_h/2, v, ha='center', va='center',
                       fontsize=8, color=color, fontweight=fw,
                       bbox=dict(boxstyle='square,pad=0.3',
                                 facecolor=bg, edgecolor='#E0E0E0', linewidth=0.4))
            x += w

    legend_patches = [mpatches.Patch(facecolor=color, label=label, alpha=0.85)
                      for label, color in [('PASS','#2E7D32'),('MARGINAL','#F57F17'),('FAIL','#C62828')]]
    ax_sc.legend(handles=legend_patches, loc='lower right', fontsize=8, title='Status', framealpha=0.9)

    pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    # ══════════════════════════════════════════════════════════════════════════
    # PAGE 4 — TEMPERATURE SENSITIVITY
    # ══════════════════════════════════════════════════════════════════════════
    fig = plt.figure(figsize=(11.69, 8.27))
    _header(fig, 'Temperature Sensitivity', run_info, 'p.4')
    _footer(fig, COMPANY_NAME, PDF_OUT_PATH)

    gs4 = gridspec.GridSpec(2, 2, figure=fig, left=0.08, right=0.96,
                            top=0.89, bottom=0.08, hspace=0.45, wspace=0.35)

    t_vals = df_sweep['Temp (C)'].values
    depleted_mask = df_sweep['Depleted'].values
    dot_colors = ['#E53935' if d else ACCENT for d in depleted_mask]

    metrics = [
        ('Final SoC (%)',  'Final SoC (%)', '#2196F3'),
        ('Peak sag (V)',   'Peak sag (V)',  '#FF9800'),
        ('Min V (V)',      'Min V (V)',     '#E53935'),
    ]
    temp_rise = df_sweep['Max T (C)'].values - df_sweep['Temp (C)'].values

    for i, (col, ylabel, color) in enumerate(metrics):
        ax = fig.add_subplot(gs4[i // 2, i % 2])
        y  = df_sweep[col].values
        ax.plot(t_vals, y, color=color, linewidth=2)
        ax.scatter(t_vals, y, c=dot_colors, s=40, zorder=4)
        ax.set_xlabel('Ambient (°C)'); ax.set_ylabel(ylabel)
        ax.set_title(ylabel)

    ax4 = fig.add_subplot(gs4[1, 1])
    ax4.plot(t_vals, temp_rise, '#9C27B0', linewidth=2)
    ax4.scatter(t_vals, temp_rise, c=dot_colors, s=40, zorder=4)
    ax4.set_xlabel('Ambient (°C)'); ax4.set_ylabel('Self-heating (°C)')
    ax4.set_title('Cell Self-Heating')

    legend_els = [mpatches.Patch(color=ACCENT, label='Completed'),
                  mpatches.Patch(color='#E53935', label='Depleted')]
    fig.legend(handles=legend_els, loc='lower center', ncol=2, fontsize=9,
               bbox_to_anchor=(0.5, 0.01))

    pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    # ══════════════════════════════════════════════════════════════════════════
    # PAGE 5 — MULTI-BATTERY COMPARISON CHARTS
    # ══════════════════════════════════════════════════════════════════════════
    fig = plt.figure(figsize=(11.69, 8.27))
    _header(fig, 'Battery Comparison Curves', run_info, 'p.5')
    _footer(fig, COMPANY_NAME, PDF_OUT_PATH)

    gs5 = gridspec.GridSpec(2, 2, figure=fig, left=0.08, right=0.96,
                            top=0.89, bottom=0.08, hspace=0.45, wspace=0.35)
    palette = [ACCENT, '#FF9800', '#4CAF50', '#E91E63', '#9C27B0']

    axes5 = [fig.add_subplot(gs5[r, c]) for r in range(2) for c in range(2)]
    fields = [('soc_pct', 'SoC (%)'), ('voltage_v', 'Voltage (V)'),
              ('current_a', 'Current (A)'), ('temp_c', 'Temp (°C)')]

    for (field, ylabel), ax in zip(fields, axes5):
        for r, p, col in zip(compare_results, compare_packs, palette):
            t_r = np.array(r.time_s)
            y_r = np.array(getattr(r, field))
            ax.plot(t_r, y_r, color=col, linewidth=1.8,
                    label=f'{p.battery_id} ({p.pack_energy_wh:.0f}Wh)')
        ax.set_xlabel('Time (s)'); ax.set_ylabel(ylabel); ax.set_title(ylabel)

    axes5[0].legend(fontsize=7, loc='lower left')

    pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    # ══════════════════════════════════════════════════════════════════════════
    # PAGE 6 — FITTED PARAMETERS (if log available)
    # ══════════════════════════════════════════════════════════════════════════
    if fitted:
        fig = plt.figure(figsize=(11.69, 8.27))
        _header(fig, 'Reverse-Engineered Battery Parameters', run_info, 'p.6')
        _footer(fig, COMPANY_NAME, PDF_OUT_PATH)

        ax_fp = fig.add_axes([0.05, 0.10, 0.90, 0.78])
        ax_fp.axis('off')

        rows_fp = [
            ('Parameter', 'Fitted value', 'Catalog value', 'Δ vs catalog', 'R²', 'Confidence'),
            ('R_internal (mΩ)',
             f'{fitted.r_internal_mohm.value:.2f} ± {fitted.r_internal_mohm.uncertainty:.2f}' if fitted.r_internal_mohm else '—',
             f'{primary_pack.internal_resistance_mohm:.1f}',
             f'{fitted.r_internal_mohm.value - primary_pack.internal_resistance_mohm:+.1f}' if fitted.r_internal_mohm else '—',
             f'{fitted.r_internal_mohm.r_squared:.3f}' if fitted.r_internal_mohm else '—',
             'Good' if fitted.r_internal_mohm and fitted.r_internal_mohm.r_squared > 0.5 else 'Low'),
            ('Peukert k',
             f'{fitted.peukert_k.value:.4f}' if fitted.peukert_k else '—',
             '1.050', '—', '—', '—'),
            ('Capacity (Ah)',
             f'{fitted.actual_capacity_ah.value:.3f}' if fitted.actual_capacity_ah else '—',
             f'{primary_pack.pack_capacity_ah:.3f}',
             f'{fitted.actual_capacity_ah.value - primary_pack.pack_capacity_ah:+.3f}' if fitted.actual_capacity_ah else '—',
             '—', '—'),
            ('Degradation', f'{fitted.degradation_pct:.1f}%', '0%', '—', '—', '—'),
        ]

        col_x = [0.0, 0.25, 0.42, 0.57, 0.70, 0.80]
        col_w = [0.25, 0.17, 0.15, 0.13, 0.10, 0.20]
        y_fp = 0.92; row_fp = 0.11

        for ridx, row_data in enumerate(rows_fp):
            bg = DARK if ridx == 0 else ('#F5F5F5' if ridx % 2 == 0 else 'white')
            fc = 'white' if ridx == 0 else TEXT
            fw = 'bold' if ridx == 0 else 'normal'
            for cidx, (val, x, w) in enumerate(zip(row_data, col_x, col_w)):
                if ridx > 0 and cidx == 5:  # colour-code confidence
                    fc2 = '#2E7D32' if val == 'Good' else ('#F57F17' if val == 'Low' else TEXT)
                else:
                    fc2 = fc
                ax_fp.text(x + w/2, y_fp - ridx * row_fp + row_fp/2,
                           val, ha='center', va='center', fontsize=9,
                           color=fc2, fontweight=fw,
                           bbox=dict(boxstyle='square,pad=0.3',
                                     facecolor=bg, edgecolor='#E0E0E0',
                                     linewidth=0.5))

        if fitted.fit_warnings:
            wtext = '  ⚠  ' + '   ⚠  '.join(fitted.fit_warnings)
            ax_fp.text(0.5, 0.05, wtext, ha='center', va='center',
                       fontsize=8, color='#E65100',
                       bbox=dict(boxstyle='round,pad=0.4', facecolor='#FFF3E0',
                                 edgecolor='#FF9800'))

        pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

print(f'PDF report saved: {PDF_OUT_PATH}')
print(f'Pages: cover + mission + scorecard + temperature + comparison' +
      (' + fitted params' if fitted else ''))

NotImplementedError: Derived must override

## 5 · Quick Battery Selection Table

Which batteries from the catalog can safely complete this mission at all temperature sweep points?

In [ ]:
from mission.simulator import SimulationResult
all_packs = list(db.packs.values())
selection_rows = []
print(f'Testing {len(all_packs)} packs across {len(TEMP_SWEEP)} temperatures...')
for p in all_packs:
    row = {'Pack_ID': p.battery_id, 'Chemistry': p.chemistry_id,
           'Energy (Wh)': p.pack_energy_wh, 'Weight (g)': p.pack_weight_g,
           'Wh/kg': p.specific_energy_wh_kg}
    pass_all = True
    for temp in [-10, 0, 15, 25, 35]:
        try:
            r = run_simulation(p, mission, uav, db.discharge_pts,
                               ambient_temp_c=temp, dt_s=10.0)
            row[f'{temp}C SoC'] = round(r.final_soc, 1)
            row[f'{temp}C pass'] = not r.depleted and r.final_soc > 15
            if r.depleted or r.final_soc <= 15: pass_all = False
        except Exception as e:
            row[f'{temp}C SoC'] = 'ERR'; row[f'{temp}C pass'] = False
            pass_all = False
    row['All temps PASS'] = pass_all
    selection_rows.append(row)

df_sel = pd.DataFrame(selection_rows).set_index('Pack_ID')
df_sel.sort_values('All temps PASS', ascending=False)